# Diffusion-TS

In [ ]:
import os
import numpy as np
import yaml
import torch
import matplotlib.pyplot as plt

from Utils.io_utils import load_yaml_config, instantiate_from_config
from Data.build_dataloader import build_dataloader
from engine.solver import Trainer

## DataLoader

In [ ]:
from torch.utils.data import Dataset

class MyNpyDataset(Dataset):
    """
    A custom Dataset class to load your .npy format curve data.
    This class definition now lives inside the notebook.
    """
    def __init__(self, root_path, data_path, flag='train', training_ratio=0.8, **kwargs):
        full_path = os.path.join(root_path, data_path)
        all_data = np.load(full_path)
        
        train_len = int(len(all_data) * training_ratio)
        
        if flag == 'train':
            self.data = all_data[:train_len]
        else:
            self.data = all_data[train_len:]
        
        self.window = self.data.shape[1]
        self.var_num = self.data.shape[2]
        self.auto_norm = False
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]

In [ ]:
# --- Configuration for data processing ---
SOURCE_DATA_PATH = "./Bamboo500.npy"
OUTPUT_DIR = "Data/my_curves_notebook"
DATA_FILENAME = "my_curves_data.npy"

def process_and_save_data(source_path, out_dir, out_filename):
    """
    Loads the raw 2D data, converts it to float32, and adds a feature dimension.
    """
    print("--- Starting data preprocessing ---")
    if not os.path.exists(source_path):
        print(f"Error: Source data file not found at '{source_path}'")
        return None
        
    data_2d = np.load(source_path).astype(np.float32)
    prepared_data = np.expand_dims(data_2d, axis=-1)
    
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
        
    output_path = os.path.join(out_dir, out_filename)
    np.save(output_path, prepared_data)
    print(f"Data preprocessing complete. Saved to: {output_path}")
    return output_path

# Execute the data preprocessing
processed_data_path = process_and_save_data(SOURCE_DATA_PATH, OUTPUT_DIR, DATA_FILENAME)

## config

In [ ]:
# (Corrected Cell for YAML Configuration)

CONFIG_DIR = "Config"
CONFIG_FILENAME = "my_curves_notebook.yaml"
RESULTS_DIR = "Checkpoints_my_curves_notebook"
OUTPUT_DIR = "Data/my_curves_notebook"       # This should be defined from the previous cell
DATA_FILENAME = "my_curves_data.npy"     # This should be defined from the previous cell

if not os.path.exists(CONFIG_DIR):
    os.makedirs(CONFIG_DIR)

config_path = os.path.join(CONFIG_DIR, CONFIG_FILENAME)

# Define the complete YAML content
# This version includes all necessary parameters from your verified my_curves.yaml file.
yaml_content = f"""
model:
  target: Models.interpretable_diffusion.gaussian_diffusion.Diffusion_TS
  params:
    seq_length: 64
    feature_size: 1
    n_layer_enc: 1
    n_layer_dec: 2
    d_model: 64
    timesteps: 1000
    sampling_timesteps: 1000
    loss_type: 'l1'
    beta_schedule: 'cosine'
    n_heads: 4
    mlp_hidden_times: 4
    attn_pd: 0.0
    resid_pd: 0.0
    kernel_size: 1
    padding_size: 0

solver:
  base_lr: 1.0e-4
  max_epochs: 5000
  results_folder: ./{RESULTS_DIR}
  gradient_accumulate_every: 2
  save_cycle: 200
  ema:
    decay: 0.995
    update_interval: 10
  scheduler:
    target: engine.lr_sch.ReduceLROnPlateauWithWarmup
    params:
      factor: 0.5
      patience: 3000
      min_lr: 1.0e-6
      threshold: 1.0e-4
      threshold_mode: rel
      warmup_lr: 8.0e-4
      warmup: 500
      verbose: False

dataloader:
  train_dataset:
    target: __main__.MyNpyDataset
    params:
      root_path: '{OUTPUT_DIR}'
      data_path: '{DATA_FILENAME}'
      flag: 'train'
      training_ratio: 0.8
  
  test_dataset:
    target: __main__.MyNpyDataset
    params:
      root_path: '{OUTPUT_DIR}'
      data_path: '{DATA_FILENAME}'
      flag: 'test'
      training_ratio: 0.8

  batch_size: 64
  sample_size: 6000
  shuffle: True
"""

# Write the YAML content to the file
with open(config_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content)

# Load the config for use in later cells
config = load_yaml_config(config_path)

## train

In [ ]:
class MockArgs:
    """A class to simulate command-line arguments."""
    def __init__(self, cfg_path, results_dir):
        self.config_file = cfg_path
        self.gpu = 0
        self.train = True
        self.sample = 0
        self.name = "my_curves"
        self.save_dir = results_dir

args = MockArgs(config_path, RESULTS_DIR)

print("Building the trainer...")
torch.cuda.set_device(args.gpu)

dataloader_info = build_dataloader(config, args)
model = instantiate_from_config(config['model']).to(f'cuda:{args.gpu}')
trainer = Trainer(config=config, args=args, model=model, dataloader=dataloader_info)
print("Trainer built successfully!")
trainer.train()

## sample

In [ ]:
# last_milestone = config['solver']['max_epochs'] // config['solver']['save_cycle']
last_milestone = 20
print(f"Loading the final model checkpoint: milestone #{last_milestone}")
trainer.load(last_milestone)
num_to_generate = config['dataloader']['sample_size']
seq_len = config['model']['params']['seq_length']
feat_size = config['model']['params']['feature_size']

train_dataset = dataloader_info['dataset']

print(f"Proceeding to generate {num_to_generate} new samples...")
generated_samples = trainer.sample(num=len(train_dataset), size_every=num_to_generate, shape=[seq_len, feat_size])

if isinstance(generated_samples, torch.Tensor):
    generated_samples = generated_samples.cpu().numpy()

if generated_samples.shape[-1] == 1:
    generated_samples = generated_samples.squeeze(-1)

print(f"--- Sample generation complete. Generated array shape: {generated_samples.shape} ---")

output_directory = args.save_dir
os.makedirs(output_directory, exist_ok=True)
print(f"Ensured output directory exists: {output_directory}")

output_filename = "vectors_diffusion-TS.npy"
output_path = os.path.join(output_directory, output_filename)
np.save(output_path, generated_samples)

In [23]:
generated_samples = np.load('vectors_diffusion-TS.npy')
generated_result = generated_samples[:6000].reshape(-1, 64)
np.save('generated_vectors_diffusion-TS.npy', generated_result)

## Erosion process
erosion process  

In [3]:
class FractureCurveGenerator:
    def __init__(self, count_fiber, K_Ⅲ, len_x, ce_rate, erosion_epoch, data_file="generated_vectors.npy", save_path='seriesgan_data_6000.npy'):
        self.count_fiber = count_fiber
        self.K_Ⅲ = K_Ⅲ
        self.len_x = len_x
        self.ce_rate = ce_rate
        self.erosion_epoch = erosion_epoch
        self.save_path = save_path
        self.data_file = data_file
        
        # Load pre-generated data
        self.generated_samples = np.load(data_file)
        self.current_index = 0  # Track current data index being used
        
        print(f"Loaded {len(self.generated_samples)} samples from {data_file}")
        print(f"Sample shape: {self.generated_samples.shape}")

    def create_fiber_line(self): 
        '''Description: This function is used to generate the fracture curve'''
        # Get a sample from pre-loaded data
        if self.current_index >= len(self.generated_samples):
            # If all data is used up, start over with cycling
            self.current_index = 0
            print("Warning: Reusing data samples as all samples have been used.")
        
        # Get current sample
        sample = self.generated_samples[self.current_index]
        self.current_index += 1
        
        # Convert to list format
        if isinstance(sample, np.ndarray):
            y_list = sample.tolist()
        else:
            y_list = [sample] if not isinstance(sample, list) else sample
            
        return y_list

    def reset_data_index(self):
        '''Reset the data index to start from the beginning'''
        self.current_index = 0

    def get_random_fiber_line(self):
        '''Get a random sample from the loaded data'''
        random_index = np.random.randint(0, len(self.generated_samples))
        sample = self.generated_samples[random_index]
        
        if isinstance(sample, np.ndarray):
            y_list = sample.tolist()
        else:
            y_list = [sample] if not isinstance(sample, list) else sample
            
        return y_list

    def relu(self, x):
        '''Calculate the ReLU activation function'''
        return np.maximum(0, x)

    def erosion_new(self, list):
        '''Generate a single step of erosion on the curve'''
        new_list = []
        for i in range(len(list)):
            # Determine left and right neighbor values
            if i == 0:
                left_fiber = list[i]
                if len(list) == 1:
                    right_fiber = list[i]
                else:
                    right_fiber = list[i+1]
            elif i == len(list)-1:
                left_fiber = list[i-1]
                right_fiber = list[i]
            else:
                left_fiber = list[i-1]
                right_fiber = list[i+1]
            
            # Calculate erosion for each fiber element
            structural_area = self.relu(list[i]-left_fiber)+self.relu(list[i]-right_fiber)  # Calculate structural area
            erosion_fiber = list[i]-structural_area*self.ce_rate
            new_list.append(erosion_fiber)

        return new_list

    def erosion_with_epoch(self, list, epoch): 
        '''Apply erosion process for multiple epochs'''
        for i in range(epoch):
            list = self.erosion_new(list)
        return list

    def floor_list(self, list, floor):
        '''Adjust the list values to a specified floor level'''
        min_value = min(list) + floor
        adjusted_list = [x - min_value for x in list]
        return adjusted_list

    def revers_list(self, list, floor): 
        '''Reverse the list values and adjust to floor level'''
        inverted_list = [-x for x in list]
        return self.floor_list(inverted_list, floor)

    def get_top_erosion_fiber(self, list, epoch, floor): 
        '''Generate the top erosion fiber through reverse processing'''
        reversed_list = self.revers_list(list, 0)
        erosioned_list = self.erosion_with_epoch(reversed_list, epoch)
        adjusted_list = self.revers_list(erosioned_list, -floor)
        return adjusted_list

    def fibers_resize(self, list, num):
        '''Resize fibers to specified number of points using interpolation'''
        # Create new indices with specified length
        new_indices = np.linspace(0, len(list) - 1, num=num)
        # Use linear interpolation
        resampled_list = np.interp(new_indices, np.arange(len(list)), list)
        resampled_list = [64*x/self.count_fiber for x in resampled_list]
        resampled_list = [round(x, 4) for x in resampled_list]
        return resampled_list

    def get_pair_fibers(self, use_random=False):
        '''Generate a pair of erosion fibers (top and bottom)'''
        if use_random:
            fracture_list = self.get_random_fiber_line()
        else:
            fracture_list = self.create_fiber_line()
            
        erosion_list = self.erosion_with_epoch(fracture_list, self.erosion_epoch)
        erosion_list = self.floor_list(erosion_list, 0)
        top_erosion_list = self.get_top_erosion_fiber(fracture_list, self.erosion_epoch, 0)
        return self.fibers_resize(erosion_list, 64), self.fibers_resize(top_erosion_list, 64)

    def get_fracture_curves(self, data_amount):
        '''Generate processed fracture curve data for training'''
        data_list = []
        array_zero = np.zeros(64)
        
        # Reset index to start from the beginning
        self.reset_data_index()
        
        # Generate basic fracture curve pairs
        for i in range(data_amount):
            a, b = self.get_pair_fibers()
            vector_edge_top = np.array(a)
            vector_edge_bottom = np.array(b)
            list_top_bottom = [array_zero, array_zero, vector_edge_top, vector_edge_bottom]
            data_list.append(list_top_bottom)

        all_data_list = np.array(data_list)
        np.save(self.save_path, all_data_list)
        print(f"Saved {len(all_data_list)} processed samples to '{self.save_path}'")
        return all_data_list

In [41]:
generator = FractureCurveGenerator(
    count_fiber=64,  # based on the dimension of generated_vectors.npy
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.02,
    erosion_epoch=500,
    data_file="generated_vectors_diffusion-TS.npy",
    save_path='generated_vectors_diffusion-TS_0.02_6000.npy'
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_diffusion-TS.npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_diffusion-TS_0.02_6000.npy'


In [42]:
generator = FractureCurveGenerator(
    count_fiber=64,  # based on the dimension of generated_vectors.npy
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.01,
    erosion_epoch=500,
    data_file="generated_vectors_diffusion-TS.npy",
    save_path='generated_vectors_diffusion-TS_0.01_6000.npy'
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_diffusion-TS.npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_diffusion-TS_0.01_6000.npy'


In [4]:
def perturb_curve(y, noise_level=0.02):
    """
    Simplified curve perturbation function to add realistic roughness
    
    Args:
        y: Original curve data (1D numpy array, float32 format)
        noise_level: Overall noise strength (0.01-0.1 recommended)
    
    Returns:
        Perturbed curve with added roughness (same format as input)
    """
    # Ensure input is numpy array and preserve original dtype
    y = np.asarray(y)
    original_dtype = y.dtype
    
    x = np.linspace(0, 1, len(y))
    
    # 1. Basic Gaussian noise
    basic_noise = np.random.normal(0, noise_level, len(y))
    
    # 2. High-frequency noise for fine-scale variations
    high_freq = np.random.normal(0, noise_level * 0.5, len(y))
    
    # 3. Mid-frequency periodic perturbation
    mid_freq = np.sin(2*np.pi*5*x) * noise_level * 2
    
    # 4. Low-frequency drift for long-term variations
    low_freq = np.cumsum(np.random.normal(0, noise_level/10, len(y)))
    low_freq = low_freq - np.linspace(low_freq[0], low_freq[-1], len(low_freq))
    
    # 5. Nonlinear perturbation: y_new = y + β * sin(γ * y) * noise
    nonlinear_noise = np.random.normal(0, noise_level * 0.8, len(y))
    nonlinear = 0.03 * np.sin(2 * y) * nonlinear_noise
    
    # Combine all perturbations
    y_perturbed = y + basic_noise + high_freq + mid_freq + low_freq + nonlinear
    
    # Preserve original dtype (float32)
    return y_perturbed.astype(original_dtype)

In [5]:
generated_samples = np.load("generated_vectors_diffusion-TS.npy")
perturbed_samples = [perturb_curve(sample, noise_level=0.12) for sample in generated_samples]
np.save("generated_vectors_diffusion-TS_perturbed.npy", perturbed_samples)

In [6]:
generator = FractureCurveGenerator(
    count_fiber=64,  
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.02,
    erosion_epoch=500,
    data_file="generated_vectors_diffusion-TS_perturbed.npy",
    save_path="generated_vectors_diffusion-TS_perturbed_0.02_6000.npy"
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_diffusion-TS_perturbed.npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_diffusion-TS_perturbed_0.02_6000.npy'
